# Backend

## HTTP
Es un protocolo **stateless**, no hay vínculo entre dos solicitudes, aunque sean consecutivas y del mismo cliente (responde y se olvida).
Se desentiende de la capa de transporte, confía en TPC para ello.

Secciones del protocolo http:
- Header
- (Línea sepadarora)
- Body

\* Las cabeceras con instrucciones para el browser.

\* Si un hacker inyecta código desde el cliente, el browser lo filtra fácilmente (WAF, Web Application Firewall), lo peligroso es cuando inyecta al código fuente, porque el browser no podrá diferenciar el código legítimo del maligno.

### Request
metodo_del_request URI version-HTTP

- método del request: GET, POST, HEAD, etc.
- URI: (NO URL) especifica el recurso que se requiere.
- versión HTTP.

#### Métodos para request
\* Similar a CRUD
- GET: solicitud del cliente para obtener el recurso del servidor (se usa por default).
- HEAD: retorna sólo los headers (no el body).
- POST: __crea__ un nuevo recurso, no es idempotente (puede crear múltiples recursos si se llama varias veces).
- PUT: solicita la __actualización__ de un recurso existente, idempotente (puede ser llamado múltiples veces sin cambiar el resultado).
- PATCH: solicita la __actualización parcial__ de un recurso existente, idempotente.
- DELETE: borrar datos del servidor.
- OPTIONS: pide un listado con los métodos del request que el server soporta/implementa.
- CONNECT: se usa para establecer conexiones seguras vía SSL, se solicita realizar la conexión hacia el server, y retorna la información, pero no se hace nada con ella.

##### Estructura
METHOD/path HTTP-version
Headers <!--HOST: request header, Content-Type:  entity header -->
<!-- Blanck line (separación) -->
Body <!-- JSON -->

#### Response
HTTP-version status explicación

- HTTP-version
- Status: código de 3 dígitos que indica el estado de la request.
- Explicación: pequeña explicación del status.

\* Los headers tienen formato llave:valor

##### Cógidos de respuesta:
- 1XX: Solicitud recibida
- 2XX: Solicitud recibida exitosamente
- 3XX: Redireccionamiento (para completar la solicitud se debe tomar alguna acción)
- 4XX: Client-side error (errores desde el origen de la solicitud)
- 5XX: Server-side error (el servidor lo pudo responder lo que parece ser una solicitud válida)


##### curl
Comando para hacer solicitudes HTTP desde la terminal.
Flags:
- -X: especifica el método HTTP a usar (GET, POST, PUT, DELETE, etc.).
- -H: agrega un header a la solicitud (IF-NONE-MATCH, etc).
- -d: envía datos en el cuerpo de la solicitud (POST, PUT).
- -I: obtiene sólo los headers de la respuesta (HEAD).
- -v: modo verbose, muestra detalles de la solicitud y respuesta.
- -k: permite conexiones inseguras (ignora errores SSL).

Ejemplo:
```bash
curl https://api.example.com/resource
curl -X POST https://api.example.com/resource -H "Content-Type: application/json" -d '{"key":"value"}'
curl -I https://api.example.com/resource
curl -X DELETE https://api.example.com/resource/123
```


## Middleware
\* Es la parte del controlador en MVC.

Es una pieza de software que reside entre el cliente y el servidor. Suele manejar autenticación, validación, almacenamiento de transacciones, etc. en lo que se conoce como ciclo request-response.
\* Para atender todos los requerimientos de un cliente, se encadenan requests en capas encadenadas.

#### KOA
Es un framework de Node.js que se ejecuta entre medio del proceso request/response (middleware).
\* Una aplicación koa puede tener muchos middlewares que se ejecutan en un orden determinado (cascada).

_Una app koa es es un objeto que contiene un array de funciones middleware que se disponen y ejecutan en forma de stack cuando son requeridas._

Cuando un middleware llama `next()`, la función del middleware se suspende (await) y pasa el control al siguiente middleware definido (pasa al siguiente app.use).
Cuando ya no hay más middlewares para ejecutarse, el control se desenrolla y cada middleware reanuda su ejecución "aguas arriba".


In [ ]:
const Koa = require('koa');
const app = new Koa();

app.method('path', async ctx => {
    // ...
})

app.use(async ctx => { // .use registra un middleware
    ctx; // contexto
    ctx.request; // instancia del objeto Koa request, da acceso a métodos y propiedades del request
    ctx.response; // instancia del objeto Koa response, da acceso a métodos y propiedades del response
    ctx.state; // objeto para guardar información y compartirla entre middlewares (ej: .user recupera datos del usuario)
})

app.use(async (ctx, next) => { // next es una función que llama al siguiente middleware
    // ... código antes de llamar al siguiente middleware
    await next(); // llama al siguiente middleware
    // ... código después de que el siguiente middleware y los siguientes hayan terminado
})

// ...
// Se pueden concatenar middlewares mediante next() para crear un stack de middlewares (en cascada)

app.listen(port);

Existen diferentes módulos para koa:
- koa-router: para definir rutas.
- koa-bodyparser: para parsear el body de las requests.
- koa-etag: para manejar el cacheo.
- koa-logger: para loguear requests.
- koa-session: para manejar sesiones.

# API REST
Application Programming Interface, Representational State Transfer

Es una interfaz que permite la interacción entre múltiples "actores" del software, la forma en la que los trozos de software distribuidos se comunican entre sí.

\* Hay muchos tipos de APIs.

Una API en sencillo tiene rutas (endpoints) y verboos HTTP (GET, POST, PUT, DELETE, etc). Con eso la API puede dirigir las solicitudes a los controladores que se encargan de la lógica de negocio, los verbos dicen qué hacer, y las rutas dónde hacerlo.

\* Hay validaciones que se pueden hacer en el frontend, con eso nos ahorramos tráfico de red y procesamiento en el backend. Pero las validaciones en el backend son obligatorias. Separar lógica de cliente y servidor.

\* REST es un estilo arquitectónico, no un protocolo.


\* Las cookies sirven para saber en qué estado estoy (porque HTTP es stateless).
\* Cookies es un tracker (se llama cookies por el "rastro de migas" que deja una galleta).

\* JWT (JSON Web Token)

\* OAuth (Open Authorization)

### Cookies

Son pequeñas piezas de información que sirven para sacar HTTP de su estado stateless.
- Son enviadas por el server al browser en la response (se envía como parte del header).
- Es texto plano
- Se almacenan en el browser.
- Sólo los sitios que mandaron las cookies pueden leerlas, no terceros.
- No pueden exceder los 4KB.
- Los dominios pueden mandar 20 a 25 cookies como límite.

```Plain
HTTP/1.1 200 OK
Content-Type: text/html
Content-Length:
Set-Cookie: items=16
Set-Cookie: screenmode=dark,Expires=Sun, 1 Jan 2023 12:00:00 GMT
Set-Cookie: Max-Age: 3600; Secure; HttpOnly
```

\* El servidor envía cookie sólo una vez para settear la sesión, luego es el browser el que la envía cada vez para autenticarse.
### JWT
JSON Web Token

Es un estándar abierto que describe un método para estructurar el intercambio de _tokens_ entre dos sistemas.

- Colocar información precisa y breve en el payload. Las solicitudes deben ser compactas.
- **NUNCA** colocar información sensible en JWT, son fácilemente decodificables.
- No pretenden encriptar información, sólo velar por la integridad y autenticidad de los datos.
- Están codificados y firmados, NO cifrados.
- Da seguridad a la a la información de los headers de autenticación.

Estructura:
```Plain
AlgoritmoDeEncriptacion.Payload.Firma
```

- Algoritmo de encriptación: algoritmo usado para firmar el token (HMAC, SHA256, RSA, etc).
- Payload: información para transferir en formato JSON.
- Firma: se obtiene codificando el header y payload en codificación Base64url.

Funcionamiento:
1. Usuario se loguea con user y pwd.
2. Server autentifica al usuario y le entrega un JWT firmado con la firma secreta que conoce el server.
3. El cliente usa el JWT para acceder a recursos protegidos.
4. Cada vez que se accede a recursos protegidos, el server verifica la autenticidad del token con la llave privada.

### Koa-session
Módulo de koa para manerjar sesiones.
Reconoce específicamente a un usuario en requests sucesivas usando cookies.

In [ ]:
import session from 'koa-session';
import koa from 'koa';
const app = new koa();
app.keys = ['La palabra secreta'];
app.use(session(app)) ;
app.use ( ctx => { ctx.session;
    let n = ctx.session.views || 0;
    ctx.session.firstKey = 'firstValue';
    console.log(ctx.session);
    ctx.session.views = ++n;
    ctx.body = "hola mundo" + n
})
app.listen(3000);

### OAuth 2.0
Es un conjunto de especificaciones que permiten a los desarrolladores delegar fácilmente en _otra entidad_ de autenticación y autorización de sus usuarios.

Se envía un token de acceso a un servidor externo que autentica al usuario que retorna un token de acceso al cliente, que a su vez lo envía al servidor de recursos para acceder a los datos protegidos.

##### SSO
Single Sign On
Es un método de autenticación que permite a los usuarios autenticarse de forma segura en _varias aplicaciones_ utilizando un _único conjunto de credenciales_.
Funciona sobre la relación de confianza establecida entre la aplicación y el proveedor de identidades.

##### OpenID
Es un estándar _abierto_ que permite la identidad digital descentralizada, permitiendo a los usuarios iniciar sesión en diferentes sitios web usando el mismo proveedor de identidad.

### WebSockets
Permiten enviar actualizaciones en tiempo real entre el cliente y el servidor sin la necesidad de realizar solicitudes HTTP repetidas.
